# Aethermoore Art Style LoRA Training
**Trigger word:** `sixtongues_style`
**Base model:** FLUX.1-schnell
**Dataset:** 25 images from `issdandavis/six-tongues-art-style`

Run all cells top to bottom. Total time ~2-3 hours on T4.

In [ ]:
# Cell 1: Check GPU and install dependencies
!nvidia-smi
!pip install -q torch torchvision
!pip install -q diffusers transformers accelerate peft bitsandbytes
!pip install -q datasets huggingface_hub safetensors
!pip install -q prodigyopt wandb
print('\n✅ Dependencies installed')

In [ ]:
# Cell 2: Login to HuggingFace
import os
from huggingface_hub import login
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = input("Enter your HuggingFace token: ")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Done - logged in")


In [ ]:
# Cell 3: Download training images from HF dataset
from huggingface_hub import snapshot_download
import shutil
from pathlib import Path

# Download the art style dataset
dataset_path = snapshot_download(
    'issdandavis/six-tongues-art-style',
    repo_type='dataset',
    local_dir='./training_data'
)

# Check what we got
data_dir = Path('./training_data')
images = list(data_dir.rglob('*.png')) + list(data_dir.rglob('*.jpg'))
print(f'Downloaded {len(images)} images')
for img in images[:5]:
    print(f'  {img.name} ({img.stat().st_size//1024}KB)')

In [ ]:
# Cell 4: Prepare dataset with captions
import json
from pathlib import Path
from PIL import Image

data_dir = Path('./training_data')
output_dir = Path('./lora_dataset')
output_dir.mkdir(exist_ok=True)

# Load metadata if exists
meta_path = data_dir / 'metadata.jsonl'
captions = {}
if meta_path.exists():
    for line in meta_path.read_text().splitlines():
        if line.strip():
            r = json.loads(line)
            captions[r.get('file_name', '')] = r.get('text', '')
    print(f'Loaded {len(captions)} captions from metadata.jsonl')

# Process images: resize to 512x512, save with caption files
processed = 0
for img_path in sorted(data_dir.rglob('*.png')) + sorted(data_dir.rglob('*.jpg')):
    if img_path.parent.name == 'lora_dataset':
        continue
    
    # Resize to 512x512 (center crop)
    with Image.open(img_path) as im:
        im = im.convert('RGB')
        # Center crop to square
        w, h = im.size
        side = min(w, h)
        left = (w - side) // 2
        top = (h - side) // 2
        im = im.crop((left, top, left + side, top + side))
        im = im.resize((512, 512), Image.LANCZOS)
        
        out_name = f'{processed:03d}'
        im.save(output_dir / f'{out_name}.png')
    
    # Write caption
    caption = captions.get(img_path.name, '')
    if not caption:
        caption = f'sixtongues_style, dark fantasy illustration in manhwa style'
    if 'sixtongues_style' not in caption:
        caption = 'sixtongues_style, ' + caption
    
    (output_dir / f'{out_name}.txt').write_text(caption)
    processed += 1

print(f'\n✅ Processed {processed} image+caption pairs to {output_dir}')
print(f'Sample caption: {(output_dir / "000.txt").read_text()[:100]}...')

In [ ]:
# Cell 5: Configure LoRA training
import torch

# Training config
CONFIG = {
    'pretrained_model': 'black-forest-labs/FLUX.1-schnell',
    'dataset_dir': './lora_dataset',
    'output_dir': './lora_output',
    'trigger_word': 'sixtongues_style',
    
    # LoRA params
    'lora_rank': 64,          # 64 for T4 (128 needs more VRAM)
    'lora_alpha': 64,
    
    # Training params
    'learning_rate': 4e-4,
    'train_batch_size': 1,
    'gradient_accumulation_steps': 4,
    'max_train_steps': 1000,   # 25 images * 40 steps/image
    'resolution': 512,
    'mixed_precision': 'bf16' if torch.cuda.is_bf16_supported() else 'fp16',
    'seed': 42,
    'lr_scheduler': 'cosine',
    'lr_warmup_steps': 100,
}

print('Training config:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'\nGPU memory: {gpu_mem:.1f}GB')
if gpu_mem < 15:
    print('⚠️  T4 has 15GB — will use 8-bit quantization')
    CONFIG['use_8bit'] = True
else:
    CONFIG['use_8bit'] = False
    CONFIG['lora_rank'] = 128  # Can use higher rank with more VRAM

In [ ]:
# Cell 6: Train LoRA
# Using diffusers train_dreambooth_lora_flux.py approach
import os
from pathlib import Path

os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Write a training dataset in the format diffusers expects
dataset_dir = Path(CONFIG['dataset_dir'])
images = sorted(dataset_dir.glob('*.png'))
print(f'Training on {len(images)} images...')

from diffusers import FluxPipeline, FluxTransformer2DModel
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch
from transformers import CLIPTokenizer, T5TokenizerFast

class LoRADataset(Dataset):
    def __init__(self, data_dir, resolution=512):
        self.data_dir = Path(data_dir)
        self.images = sorted(self.data_dir.glob('*.png'))
        self.transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.LANCZOS),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        caption_path = img_path.with_suffix('.txt')
        
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        
        caption = caption_path.read_text().strip() if caption_path.exists() else 'sixtongues_style'
        
        return {'pixel_values': image, 'caption': caption}

dataset = LoRADataset(CONFIG['dataset_dir'], CONFIG['resolution'])
print(f'Dataset: {len(dataset)} samples')
print(f'Sample caption: {dataset[0]["caption"][:80]}...')

# Load pipeline with quantization for T4
print('\nLoading FLUX.1-schnell (this takes a few minutes)...')
pipe = FluxPipeline.from_pretrained(
    CONFIG['pretrained_model'],
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    token=HF_TOKEN,
)

# Apply LoRA to the transformer
lora_config = LoraConfig(
    r=CONFIG['lora_rank'],
    lora_alpha=CONFIG['lora_alpha'],
    target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'],
    lora_dropout=0.0,
)

transformer = pipe.transformer
transformer = get_peft_model(transformer, lora_config)
transformer.print_trainable_parameters()
transformer.to('cuda')

# Training loop
from torch.optim import AdamW
from tqdm.auto import tqdm

optimizer = AdamW(transformer.parameters(), lr=CONFIG['learning_rate'])
dataloader = DataLoader(dataset, batch_size=CONFIG['train_batch_size'], shuffle=True)

print(f'\nStarting training for {CONFIG["max_train_steps"]} steps...')
transformer.train()
global_step = 0
losses = []

pbar = tqdm(total=CONFIG['max_train_steps'], desc='Training')
while global_step < CONFIG['max_train_steps']:
    for batch in dataloader:
        if global_step >= CONFIG['max_train_steps']:
            break
        
        # Forward pass through the pipeline's components
        pixel_values = batch['pixel_values'].to('cuda', dtype=torch.bfloat16)
        captions = batch['caption']
        
        # Encode text
        with torch.no_grad():
            text_inputs = pipe.tokenizer(
                captions, padding='max_length', max_length=77,
                truncation=True, return_tensors='pt'
            ).to('cuda')
            text_embeds = pipe.text_encoder(text_inputs.input_ids)[0]
            
            # Encode images to latents
            latents = pipe.vae.encode(pixel_values).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor
        
        # Add noise
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps,
                                  (latents.shape[0],), device='cuda')
        noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
        
        # Predict noise
        model_pred = transformer(
            hidden_states=noisy_latents,
            encoder_hidden_states=text_embeds,
            timestep=timesteps,
        ).sample
        
        # Loss
        loss = torch.nn.functional.mse_loss(model_pred, noise)
        loss.backward()
        
        if (global_step + 1) % CONFIG['gradient_accumulation_steps'] == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        losses.append(loss.item())
        global_step += 1
        pbar.update(1)
        pbar.set_postfix(loss=f'{loss.item():.4f}')
        
        if global_step % 100 == 0:
            avg_loss = sum(losses[-100:]) / min(100, len(losses))
            print(f'Step {global_step}: avg_loss={avg_loss:.4f}')

pbar.close()
print(f'\n✅ Training complete! Final avg loss: {sum(losses[-50:]) / 50:.4f}')

In [ ]:
# Cell 7: Save LoRA weights
from safetensors.torch import save_file
from pathlib import Path

output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(exist_ok=True)

# Extract LoRA weights
lora_state_dict = {}
for name, param in transformer.named_parameters():
    if 'lora' in name:
        lora_state_dict[name] = param.cpu()

# Save as safetensors
lora_path = output_dir / 'aethermoore-art-lora.safetensors'
save_file(lora_state_dict, str(lora_path))
size_mb = lora_path.stat().st_size / (1024 * 1024)
print(f'✅ Saved LoRA: {lora_path} ({size_mb:.1f}MB)')
print(f'   Trainable params: {len(lora_state_dict)}')

In [ ]:
# Cell 8: Test generation
from diffusers import FluxPipeline
import torch

# Load fresh pipeline and apply LoRA
pipe = FluxPipeline.from_pretrained(
    CONFIG['pretrained_model'],
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
).to('cuda')

pipe.load_lora_weights(str(Path(CONFIG['output_dir']) / 'aethermoore-art-lora.safetensors'))

# Test prompts
test_prompts = [
    'sixtongues_style, a dark crystal cavern with glowing energy beams and sacred tongue orbs floating in formation',
    'sixtongues_style, a young man in a hoodie standing before a massive archive filled with glowing scrolls, a raven perched on his shoulder',
    'sixtongues_style, six colored orbs arranged in a circle on stone pedestals, spiral inscriptions glowing on the floor, cathedral-like chamber',
    'sixtongues_style, floating islands connected by energy bridges above clouds, crystal spires reaching into a purple sky',
]

for i, prompt in enumerate(test_prompts):
    print(f'\nGenerating test {i+1}/4: {prompt[:60]}...')
    image = pipe(
        prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        width=512,
        height=512,
    ).images[0]
    
    out_path = f'lora_output/test_{i+1}.png'
    image.save(out_path)
    print(f'  Saved: {out_path}')

print('\n✅ Test images generated! Check lora_output/test_*.png')

In [ ]:
# Cell 9: Upload to HuggingFace
from huggingface_hub import HfApi, upload_file

api = HfApi()
repo_id = 'issdandavis/aethermoore-art-lora'

# Create repo if needed
try:
    api.create_repo(repo_id, repo_type='model', exist_ok=True)
except Exception as e:
    print(f'Repo exists or error: {e}')

# Upload LoRA weights
upload_file(
    path_or_fileobj='lora_output/aethermoore-art-lora.safetensors',
    path_in_repo='aethermoore-art-lora.safetensors',
    repo_id=repo_id,
    repo_type='model',
)

# Upload test images
for i in range(1, 5):
    test_path = f'lora_output/test_{i}.png'
    if Path(test_path).exists():
        upload_file(
            path_or_fileobj=test_path,
            path_in_repo=f'samples/test_{i}.png',
            repo_id=repo_id,
            repo_type='model',
        )

# Upload model card
model_card = '''---
tags:
  - flux
  - lora
  - art-style
  - fantasy
  - manhwa
base_model: black-forest-labs/FLUX.1-schnell
license: cc-by-nc-4.0
---

# Aethermoore Art Style LoRA

FLUX.1-schnell LoRA fine-tuned on 25 concept art images from the Six Tongues Protocol / Aethermoore universe.

**Trigger word:** `sixtongues_style`

## Usage
```python
from diffusers import FluxPipeline
pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell")
pipe.load_lora_weights("issdandavis/aethermoore-art-lora")
image = pipe("sixtongues_style, crystal cavern with sacred tongue orbs").images[0]
```

## Style
Dark fantasy + cyberpunk isekai aesthetic. Manhwa/webtoon influenced.
Sacred geometry, glowing orbs, crystal structures, spiral inscriptions.

## Training
- Base: FLUX.1-schnell
- Images: 25
- Steps: 1000
- LoRA rank: 64
- Resolution: 512x512
'''

with open('lora_output/README.md', 'w') as f:
    f.write(model_card)

upload_file(
    path_or_fileobj='lora_output/README.md',
    path_in_repo='README.md',
    repo_id=repo_id,
    repo_type='model',
)

print(f'\n✅ Uploaded to https://huggingface.co/{repo_id}')
print('Done! Your LoRA is live.')